# Fused residual RMSNorm
Original CUDA kernels, FP32 residual sum and accumulation, FP16/FP32 output. Select T4 and run all. Timings include Python launch submission overhead; see README.


In [ ]:
import pathlib, subprocess, json
def run(args, log):
 p=subprocess.run(args,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
 print(p.stdout,flush=True)
 pathlib.Path(log).write_text(p.stdout)
 if p.returncode: raise RuntimeError(f"exit {p.returncode}: {args}")
run(["nvidia-smi"],"gpu.log")
run(["nvcc","--version"],"compiler.log")


In [ ]:
pathlib.Path("kernels.cu").write_text("#include <cuda_runtime.h>\n#include <cuda_fp16.h>\n#include <cmath>\n#include <cstdint>\n\n// All variants retain the residual sum in FP32 and round only the final output.\n// Each block owns a row; every lane participates in the reduction, including tails.\ntemplate <class T> __device__ float read_value(T x) { return float(x); }\ntemplate <> __device__ float read_value<half>(half x) { return __half2float(x); }\ntemplate <class T> __device__ T write_value(float x) { return T(x); }\ntemplate <> __device__ half write_value<half>(float x) { return __float2half_rn(x); }\n\n__device__ float warp_sum(float x) {\n  for (int delta = 16; delta; delta >>= 1)\n    x += __shfl_down_sync(0xffffffffu, x, delta);\n  return x;\n}\n\ntemplate <int Threads> __device__ float row_sum(float x) {\n  __shared__ float partial[Threads / 32];\n  __shared__ float total;\n  x = warp_sum(x);\n  if ((threadIdx.x & 31) == 0) partial[threadIdx.x / 32] = x;\n  __syncthreads();\n  if (threadIdx.x < 32) {\n    x = threadIdx.x < Threads / 32 ? partial[threadIdx.x] : 0.0f;\n    x = warp_sum(x);\n    if (threadIdx.x == 0) total = x;\n  }\n  __syncthreads();\n  return total;\n}\n\ntemplate <class T, int Threads>\n__global__ void fused(const T* x, const T* residual, const T* weight,\n                      T* out, int width, float eps) {\n  const int64_t base = int64_t(blockIdx.x) * width;\n  float sum = 0.0f;\n  for (int col = threadIdx.x; col < width; col += Threads) {\n    const float z = read_value(x[base + col]) + read_value(residual[base + col]);\n    sum += z * z;\n  }\n  const float scale = rsqrtf(row_sum<Threads>(sum) / float(width) + eps);\n  // Reloading avoids a width-dependent register array and supports arbitrary tails.\n  for (int col = threadIdx.x; col < width; col += Threads) {\n    const float z = read_value(x[base + col]) + read_value(residual[base + col]);\n    out[base + col] = write_value<T>((z * scale) * read_value(weight[col]));\n  }\n}\n\ntemplate <class T>\n__global__ void residual_add(const T* x, const T* residual, float* scratch, int64_t n) {\n  const int64_t i = int64_t(blockIdx.x) * blockDim.x + threadIdx.x;\n  if (i < n) scratch[i] = read_value(x[i]) + read_value(residual[i]);\n}\n\ntemplate <class T>\n__global__ void normalize(const float* scratch, const T* weight, T* out,\n                          int width, float eps) {\n  const int64_t base = int64_t(blockIdx.x) * width;\n  float sum = 0.0f;\n  for (int col = threadIdx.x; col < width; col += 256) {\n    const float z = scratch[base + col];\n    sum += z * z;\n  }\n  const float scale = rsqrtf(row_sum<256>(sum) / float(width) + eps);\n  for (int col = threadIdx.x; col < width; col += 256)\n    out[base + col] = write_value<T>((scratch[base + col] * scale) * read_value(weight[col]));\n}\n\ntemplate <class T>\ncudaError_t dispatch(const void* x, const void* residual, const void* weight,\n                     void* out, float* scratch, int rows, int width,\n                     float eps, int variant, cudaStream_t stream) {\n  const T* a = static_cast<const T*>(x);\n  const T* b = static_cast<const T*>(residual);\n  const T* w = static_cast<const T*>(weight);\n  T* y = static_cast<T*>(out);\n  if (variant == 2) {\n    const int64_t n = int64_t(rows) * width;\n    residual_add<T><<<unsigned((n + 255) / 256), 256, 0, stream>>>(a, b, scratch, n);\n    cudaError_t err = cudaGetLastError();\n    if (err != cudaSuccess) return err;\n    normalize<T><<<rows, 256, 0, stream>>>(scratch, w, y, width, eps);\n  } else if (variant == 1 && width <= 128) {\n    fused<T, 32><<<rows, 32, 0, stream>>>(a, b, w, y, width, eps);\n  } else if (variant == 1 && width <= 1024) {\n    fused<T, 128><<<rows, 128, 0, stream>>>(a, b, w, y, width, eps);\n  } else {\n    fused<T, 256><<<rows, 256, 0, stream>>>(a, b, w, y, width, eps);\n  }\n  return cudaGetLastError();\n}\n\n// C ABI accepts contiguous, nonoverlapping device buffers on the caller's stream.\n// dtype: 0=FP32, 1=FP16. variant: 0=fused256, 1=width dispatch, 2=two launches.\nextern \"C\" int rmsnorm(const void* x, const void* residual, const void* weight,\n                       void* out, float* scratch, int rows, int width,\n                       float eps, int dtype, int variant, void* stream) {\n  if (rows < 0 || width <= 0 || width > 1048576 || !std::isfinite(eps) || eps <= 0 ||\n      dtype < 0 || dtype > 1 || variant < 0 || variant > 2)\n    return int(cudaErrorInvalidValue);\n  if (rows == 0) return int(cudaSuccess);\n  if (!x || !residual || !weight || !out || (variant == 2 && !scratch))\n    return int(cudaErrorInvalidValue);\n  const int64_t n = int64_t(rows) * width;\n  if (variant == 2 && (n + 255) / 256 > 2147483647LL)\n    return int(cudaErrorInvalidConfiguration);\n  cudaStream_t s = reinterpret_cast<cudaStream_t>(stream);\n  return int(dtype == 0 ? dispatch<float>(x, residual, weight, out, scratch, rows, width, eps, variant, s)\n                       : dispatch<half>(x, residual, weight, out, scratch, rows, width, eps, variant, s));\n}\n\nextern \"C\" const char* rmsnorm_error(int error) { return cudaGetErrorString(cudaError_t(error)); }\n")
pathlib.Path("validate.py").write_text("\"\"\"Correctness and repeated CUDA-event timings for equivalent FP32-sum semantics.\"\"\"\nimport ctypes\nimport json\nimport platform\nimport statistics\nfrom pathlib import Path\n\nimport torch\n\n\nclass Kernel:\n    def __init__(self, library=\"./librmsnorm.so\"):\n        self.lib = ctypes.CDLL(str(Path(library).resolve()))\n        self.lib.rmsnorm.argtypes = [ctypes.c_void_p] * 5 + [ctypes.c_int] * 2 + [ctypes.c_float] + [ctypes.c_int] * 2 + [ctypes.c_void_p]\n        self.lib.rmsnorm.restype = ctypes.c_int\n        self.lib.rmsnorm_error.argtypes = [ctypes.c_int]\n        self.lib.rmsnorm_error.restype = ctypes.c_char_p\n\n    def launch(self, x, residual, weight, out, scratch, eps, variant):\n        if x.ndim != 2 or x.dtype not in (torch.float16, torch.float32):\n            raise ValueError(\"x must be a two-dimensional FP16 or FP32 tensor\")\n        rows, width = x.shape\n        if residual.shape != x.shape or out.shape != x.shape or weight.shape != (width,):\n            raise ValueError(\"incompatible residual, weight, or output shape\")\n        buffers = (x, residual, weight, out, scratch)\n        if any(not t.is_cuda or not t.is_contiguous() or t.device != x.device for t in buffers):\n            raise ValueError(\"buffers must be contiguous on the same CUDA device\")\n        if any(t.dtype != x.dtype for t in buffers[:4]) or scratch.dtype != torch.float32 or scratch.shape != x.shape:\n            raise ValueError(\"incompatible buffer dtype or scratch shape\")\n        if out.data_ptr() in (x.data_ptr(), residual.data_ptr(), weight.data_ptr()):\n            raise ValueError(\"output must not alias inputs\")\n        # Fresh allocations in the harness satisfy the stronger C ABI no-overlap contract.\n        with torch.cuda.device(x.device):\n            error = self.lib.rmsnorm(*(t.data_ptr() for t in buffers), rows, width, eps,\n                                     int(x.dtype == torch.float16), variant,\n                                     torch.cuda.current_stream(x.device).cuda_stream)\n        if error:\n            raise RuntimeError(self.lib.rmsnorm_error(error).decode())\n\n\ndef reference(x, residual, weight, eps):\n    z = x.float() + residual.float()\n    return (z * torch.rsqrt(z.square().mean(dim=-1, keepdim=True) + eps) * weight.float()).to(x.dtype)\n\n\ndef timing(fn, repeats=200, groups=7):\n    for _ in range(30):\n        fn()\n    torch.cuda.synchronize()\n    samples = []\n    for _ in range(groups):\n        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\n        start.record()\n        for _ in range(repeats):\n            fn()\n        end.record()\n        end.synchronize()\n        samples.append(start.elapsed_time(end) * 1000 / repeats)\n    return {\"median_us\": statistics.median(samples), \"min_us\": min(samples),\n            \"max_us\": max(samples), \"samples_us\": samples}\n\n\ndef main():\n    torch.manual_seed(3487)\n    kernel = Kernel()\n    env = {\"gpu\": torch.cuda.get_device_name(), \"capability\": torch.cuda.get_device_capability(),\n           \"torch\": torch.__version__, \"torch_cuda\": torch.version.cuda, \"python\": platform.python_version()}\n    print(json.dumps({\"environment\": env}), flush=True)\n    cases = []\n    for dtype in (torch.float32, torch.float16):\n        for rows in (1, 3, 129):\n            for width in (1, 7, 31, 32, 33, 127, 128, 129, 255, 256, 257, 1023, 1024, 1025, 4096, 4099, 8192):\n                for magnitude, eps in ((1.0, 1e-5), (1e-4, 1e-8), (0.0, 1e-5)):\n                    x = (torch.randn(rows, width, device=\"cuda\") * magnitude).to(dtype)\n                    r = (torch.randn_like(x.float()) * magnitude).to(dtype)\n                    w = torch.randn(width, device=\"cuda\", dtype=dtype)\n                    out, scratch = torch.empty_like(x), torch.empty_like(x, dtype=torch.float32)\n                    # Independent double-precision arithmetic checks accumulation and rounding.\n                    z = x.double() + r.double()\n                    expected = (z * torch.rsqrt(z.square().mean(-1, keepdim=True) + eps) * w.double()).to(dtype)\n                    for variant in (0, 1, 2):\n                        kernel.launch(x, r, w, out, scratch, eps, variant)\n                        torch.testing.assert_close(out, expected, rtol=2e-5 if dtype == torch.float32 else 2e-3,\n                                                   atol=2e-6 if dtype == torch.float32 else 2e-3)\n                    cases.append([str(dtype), rows, width, magnitude, eps])\n        # IEEE nonfinite propagation should match the equivalent framework operation.\n        for special in (float(\"nan\"), float(\"inf\"), -float(\"inf\")):\n            x = torch.ones(3, 33, device=\"cuda\", dtype=dtype)\n            x[0, 0] = special\n            r, w = torch.zeros_like(x), torch.ones(33, device=\"cuda\", dtype=dtype)\n            out, scratch = torch.empty_like(x), torch.empty_like(x, dtype=torch.float32)\n            expected = reference(x, r, w, 1e-5)\n            for variant in (0, 1, 2):\n                kernel.launch(x, r, w, out, scratch, 1e-5, variant)\n                torch.testing.assert_close(out, expected, equal_nan=True)\n    torch.cuda.synchronize()\n    print(json.dumps({\"correctness_input_cases\": len(cases), \"variant_checks\": len(cases) * 3,\n                      \"nonfinite_checks\": 18, \"status\": \"PASS\"}), flush=True)\n    results = []\n    for dtype in (torch.float32, torch.float16):\n        for rows, width in ((1, 128), (1, 4096), (32, 768), (128, 1024), (1024, 4096), (1024, 4099), (256, 8192)):\n            x = torch.randn(rows, width, device=\"cuda\", dtype=dtype)\n            r, w = torch.randn_like(x), torch.randn(width, device=\"cuda\", dtype=dtype)\n            out, scratch = torch.empty_like(x), torch.empty_like(x, dtype=torch.float32)\n            row = {\"dtype\": str(dtype), \"rows\": rows, \"width\": width, \"timing\": {}}\n            for variant, name in ((0, \"fused256\"), (1, \"width_dispatch\"), (2, \"two_kernel\")):\n                row[\"timing\"][name] = timing(lambda v=variant: kernel.launch(x, r, w, out, scratch, 1e-5, v))\n            row[\"timing\"][\"pytorch_eager\"] = timing(lambda: reference(x, r, w, 1e-5))\n            results.append(row)\n            print(json.dumps(row), flush=True)\n    Path(\"results.json\").write_text(json.dumps({\"environment\": env, \"cases\": cases, \"benchmarks\": results,\n                                               \"warmup\": 30, \"repeats\": 200, \"groups\": 7}, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n")
run(["nvcc","-std=c++17","-O3","-arch=sm_75","--fmad=false","-shared","-Xcompiler","-fPIC","kernels.cu","-o","librmsnorm.so"],"build.log")


In [ ]:
run(["python","validate.py"],"validation.log")


In [ ]:
print(json.dumps({name:pathlib.Path(name).read_text() for name in ["gpu.log","compiler.log","build.log","validation.log","results.json"]}))


In [ ]:
pathlib.Path("native.cu").write_text("#include \"kernels.cu\"\n#include <cstdio>\n#include <cstdlib>\n#include <vector>\n#include <algorithm>\n\n#define CHECK(call) do { auto e = (call); if (e != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(e)); std::exit(2); } } while (0)\n\ntemplate <class T> float host_float(T x) { return float(x); }\ntemplate <> float host_float<half>(half x) { return __half2float(x); }\ntemplate <class T> T host_value(float x) { return T(x); }\ntemplate <> half host_value<half>(float x) { return __float2half_rn(x); }\n\ntemplate <class T> void run_case(int rows, int width, bool benchmark, int only_variant = -1) {\n  const size_t n = size_t(rows) * width;\n  std::vector<T> x(n), r(n), w(width), actual(n);\n  for (size_t i = 0; i < n; ++i) {\n    x[i] = host_value<T>((int(i % 37) - 18) / 16.0f);\n    r[i] = host_value<T>((int(i % 19) - 9) / 32.0f);\n  }\n  for (int i = 0; i < width; ++i) w[i] = host_value<T>((int(i % 13) - 6) / 8.0f);\n  T *dx, *dr, *dw, *dy;\n  float* scratch;\n  CHECK(cudaMalloc(&dx, n * sizeof(T))); CHECK(cudaMalloc(&dr, n * sizeof(T)));\n  CHECK(cudaMalloc(&dw, width * sizeof(T))); CHECK(cudaMalloc(&dy, n * sizeof(T)));\n  CHECK(cudaMalloc(&scratch, n * sizeof(float)));\n  CHECK(cudaMemcpy(dx, x.data(), n * sizeof(T), cudaMemcpyHostToDevice));\n  CHECK(cudaMemcpy(dr, r.data(), n * sizeof(T), cudaMemcpyHostToDevice));\n  CHECK(cudaMemcpy(dw, w.data(), width * sizeof(T), cudaMemcpyHostToDevice));\n  const int dtype = sizeof(T) == 2;\n  for (int variant = 0; variant < 3; ++variant) {\n    if (only_variant >= 0 && only_variant != variant) continue;\n    CHECK(cudaMemset(dy, 0xff, n * sizeof(T)));\n    CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n    CHECK(cudaDeviceSynchronize());\n    CHECK(cudaMemcpy(actual.data(), dy, n * sizeof(T), cudaMemcpyDeviceToHost));\n    double max_error = 0;\n    for (int row = 0; row < rows; ++row) {\n      double sum = 0;\n      for (int col = 0; col < width; ++col) {\n        size_t i = size_t(row) * width + col;\n        double z = double(host_float(x[i])) + host_float(r[i]); sum += z * z;\n      }\n      double scale = 1.0 / std::sqrt(sum / width + double(1e-5f));\n      for (int col = 0; col < width; ++col) {\n        size_t i = size_t(row) * width + col;\n        double z = double(host_float(x[i])) + host_float(r[i]);\n        double expected = host_float(host_value<T>(float(z * scale * host_float(w[col]))));\n        double error = std::abs(double(host_float(actual[i])) - expected);\n        max_error = std::max(max_error, error);\n        double tolerance = dtype ? 0.002 + 0.002 * std::abs(expected) : 2e-6 + 2e-5 * std::abs(expected);\n        if (!std::isfinite(host_float(actual[i])) || error > tolerance) {\n          std::fprintf(stderr, \"FAIL dtype=%d rows=%d width=%d variant=%d index=%zu\\n\", dtype, rows, width, variant, i);\n          std::exit(1);\n        }\n      }\n    }\n    if (!benchmark) std::printf(\"PASS dtype=%d rows=%d width=%d variant=%d max_abs=%g\\n\", dtype, rows, width, variant, max_error);\n    if (benchmark) {\n      cudaEvent_t start, stop; CHECK(cudaEventCreate(&start)); CHECK(cudaEventCreate(&stop));\n      for (int i = 0; i < 30; ++i) CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n      CHECK(cudaDeviceSynchronize());\n      std::vector<float> samples;\n      for (int group = 0; group < 7; ++group) {\n        CHECK(cudaEventRecord(start));\n        for (int i = 0; i < 200; ++i) CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n        CHECK(cudaEventRecord(stop)); CHECK(cudaEventSynchronize(stop));\n        float ms; CHECK(cudaEventElapsedTime(&ms, start, stop)); samples.push_back(ms * 1000 / 200);\n      }\n      std::printf(\"TIMING dtype=%d rows=%d width=%d variant=%d samples_us=\", dtype, rows, width, variant);\n      for (float s : samples) std::printf(\"%.6f,\", s);\n      std::sort(samples.begin(), samples.end());\n      std::printf(\" median_us=%.6f min_us=%.6f max_us=%.6f\\n\", samples[3], samples.front(), samples.back());\n      CHECK(cudaEventDestroy(start)); CHECK(cudaEventDestroy(stop));\n    }\n  }\n  CHECK(cudaFree(dx)); CHECK(cudaFree(dr)); CHECK(cudaFree(dw)); CHECK(cudaFree(dy)); CHECK(cudaFree(scratch));\n}\n\nint main(int argc, char** argv) {\n  // A single explicit workload keeps profiler replay separate from timing loops.\n  if (argc == 6) {\n    int rows = std::atoi(argv[2]), width = std::atoi(argv[3]);\n    int dtype = std::atoi(argv[4]), variant = std::atoi(argv[5]);\n    if (rows <= 0 || rows > 4096 || width <= 0 || width > 8192 ||\n        dtype < 0 || dtype > 1 || variant < 0 || variant > 2) return 2;\n    if (dtype) run_case<half>(rows, width, false, variant);\n    else run_case<float>(rows, width, false, variant);\n    return 0;\n  }\n  bool benchmark = argc > 1;\n  if (!benchmark) {\n    for (int width : {1, 7, 31, 32, 33, 127, 128, 129, 255, 256, 257, 1023, 1024, 1025, 4096, 4099, 8192}) {\n      run_case<float>(3, width, false); run_case<half>(3, width, false);\n    }\n    std::puts(\"PASS native_checks=102\");\n  } else {\n    const int shapes[][2] = {{1,128},{1,4096},{32,768},{128,1024},{1024,4096},{1024,4099},{256,8192}};\n    for (auto& s : shapes) { run_case<float>(s[0], s[1], true); run_case<half>(s[0], s[1], true); }\n  }\n}\n")
run(["nvcc","-std=c++17","-O3","-arch=sm_75","--fmad=false","native.cu","-o","native"],"native-build.log")
run(["./native"],"native-correctness.log")
run(["compute-sanitizer","--tool","memcheck","--error-exitcode","1","./native"],"memcheck.log")
run(["compute-sanitizer","--tool","synccheck","--error-exitcode","1","./native"],"synccheck.log")
run(["./native","benchmark"],"native-timing.log")


In [ ]:
import shutil
print("profilers:", {name:shutil.which(name) for name in ["ncu","nsys"]})
print(json.dumps({name:pathlib.Path(name).read_text() for name in ["native-build.log","native-correctness.log","memcheck.log","synccheck.log","native-timing.log"]}))


In [ ]:
p=subprocess.run(["ncu","--set","basic","--launch-count","1","./native"],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
pathlib.Path("ncu.log").write_text(p.stdout)
print(p.stdout)
print("ncu_exit:",p.returncode)


In [ ]:
pathlib.Path("native.cu").write_text("#include \"kernels.cu\"\n#include <cstdio>\n#include <cstdlib>\n#include <vector>\n#include <algorithm>\n\n#define CHECK(call) do { auto e = (call); if (e != cudaSuccess) { \\\n  std::fprintf(stderr, \"%s: %s\\n\", #call, cudaGetErrorString(e)); std::exit(2); } } while (0)\n\ntemplate <class T> float host_float(T x) { return float(x); }\ntemplate <> float host_float<half>(half x) { return __half2float(x); }\ntemplate <class T> T host_value(float x) { return T(x); }\ntemplate <> half host_value<half>(float x) { return __float2half_rn(x); }\n\ntemplate <class T> void run_case(int rows, int width, bool benchmark, int only_variant = -1) {\n  const size_t n = size_t(rows) * width;\n  std::vector<T> x(n), r(n), w(width), actual(n);\n  for (size_t i = 0; i < n; ++i) {\n    x[i] = host_value<T>((int(i % 37) - 18) / 16.0f);\n    r[i] = host_value<T>((int(i % 19) - 9) / 32.0f);\n  }\n  for (int i = 0; i < width; ++i) w[i] = host_value<T>((int(i % 13) - 6) / 8.0f);\n  T *dx, *dr, *dw, *dy;\n  float* scratch;\n  CHECK(cudaMalloc(&dx, n * sizeof(T))); CHECK(cudaMalloc(&dr, n * sizeof(T)));\n  CHECK(cudaMalloc(&dw, width * sizeof(T))); CHECK(cudaMalloc(&dy, n * sizeof(T)));\n  CHECK(cudaMalloc(&scratch, n * sizeof(float)));\n  CHECK(cudaMemcpy(dx, x.data(), n * sizeof(T), cudaMemcpyHostToDevice));\n  CHECK(cudaMemcpy(dr, r.data(), n * sizeof(T), cudaMemcpyHostToDevice));\n  CHECK(cudaMemcpy(dw, w.data(), width * sizeof(T), cudaMemcpyHostToDevice));\n  const int dtype = sizeof(T) == 2;\n  for (int variant = 0; variant < 3; ++variant) {\n    if (only_variant >= 0 && only_variant != variant) continue;\n    CHECK(cudaMemset(dy, 0xff, n * sizeof(T)));\n    CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n    CHECK(cudaDeviceSynchronize());\n    CHECK(cudaMemcpy(actual.data(), dy, n * sizeof(T), cudaMemcpyDeviceToHost));\n    double max_error = 0;\n    for (int row = 0; row < rows; ++row) {\n      double sum = 0;\n      for (int col = 0; col < width; ++col) {\n        size_t i = size_t(row) * width + col;\n        double z = double(host_float(x[i])) + host_float(r[i]); sum += z * z;\n      }\n      double scale = 1.0 / std::sqrt(sum / width + double(1e-5f));\n      for (int col = 0; col < width; ++col) {\n        size_t i = size_t(row) * width + col;\n        double z = double(host_float(x[i])) + host_float(r[i]);\n        double expected = host_float(host_value<T>(float(z * scale * host_float(w[col]))));\n        double error = std::abs(double(host_float(actual[i])) - expected);\n        max_error = std::max(max_error, error);\n        double tolerance = dtype ? 0.002 + 0.002 * std::abs(expected) : 2e-6 + 2e-5 * std::abs(expected);\n        if (!std::isfinite(host_float(actual[i])) || error > tolerance) {\n          std::fprintf(stderr, \"FAIL dtype=%d rows=%d width=%d variant=%d index=%zu\\n\", dtype, rows, width, variant, i);\n          std::exit(1);\n        }\n      }\n    }\n    if (!benchmark) std::printf(\"PASS dtype=%d rows=%d width=%d variant=%d max_abs=%g\\n\", dtype, rows, width, variant, max_error);\n    if (benchmark) {\n      cudaEvent_t start, stop; CHECK(cudaEventCreate(&start)); CHECK(cudaEventCreate(&stop));\n      for (int i = 0; i < 30; ++i) CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n      CHECK(cudaDeviceSynchronize());\n      std::vector<float> samples;\n      for (int group = 0; group < 7; ++group) {\n        CHECK(cudaEventRecord(start));\n        for (int i = 0; i < 200; ++i) CHECK(cudaError_t(rmsnorm(dx, dr, dw, dy, scratch, rows, width, 1e-5f, dtype, variant, nullptr)));\n        CHECK(cudaEventRecord(stop)); CHECK(cudaEventSynchronize(stop));\n        float ms; CHECK(cudaEventElapsedTime(&ms, start, stop)); samples.push_back(ms * 1000 / 200);\n      }\n      std::printf(\"TIMING dtype=%d rows=%d width=%d variant=%d samples_us=\", dtype, rows, width, variant);\n      for (float s : samples) std::printf(\"%.6f,\", s);\n      std::sort(samples.begin(), samples.end());\n      std::printf(\" median_us=%.6f min_us=%.6f max_us=%.6f\\n\", samples[3], samples.front(), samples.back());\n      CHECK(cudaEventDestroy(start)); CHECK(cudaEventDestroy(stop));\n    }\n  }\n  CHECK(cudaFree(dx)); CHECK(cudaFree(dr)); CHECK(cudaFree(dw)); CHECK(cudaFree(dy)); CHECK(cudaFree(scratch));\n}\n\nint main(int argc, char** argv) {\n  // A single explicit workload keeps profiler replay separate from timing loops.\n  if (argc == 6) {\n    int rows = std::atoi(argv[2]), width = std::atoi(argv[3]);\n    int dtype = std::atoi(argv[4]), variant = std::atoi(argv[5]);\n    if (rows <= 0 || rows > 4096 || width <= 0 || width > 8192 ||\n        dtype < 0 || dtype > 1 || variant < 0 || variant > 2) return 2;\n    if (dtype) run_case<half>(rows, width, false, variant);\n    else run_case<float>(rows, width, false, variant);\n    return 0;\n  }\n  bool benchmark = argc > 1;\n  if (!benchmark) {\n    for (int width : {1, 7, 31, 32, 33, 127, 128, 129, 255, 256, 257, 1023, 1024, 1025, 4096, 4099, 8192}) {\n      run_case<float>(3, width, false); run_case<half>(3, width, false);\n    }\n    std::puts(\"PASS native_checks=102\");\n  } else {\n    const int shapes[][2] = {{1,128},{1,4096},{32,768},{128,1024},{1024,4096},{1024,4099},{256,8192}};\n    for (auto& s : shapes) { run_case<float>(s[0], s[1], true); run_case<half>(s[0], s[1], true); }\n  }\n}\n")
run(["nvcc","-std=c++17","-O3","-arch=sm_75","--fmad=false","native.cu","-o","native"],"native-build-final.log")
for dtype in [0,1]:
 run(["ncu","--set","basic","--launch-count","1","./native","profile","1024","4096",str(dtype),"0"],f"ncu-fused-fp{32 if dtype==0 else 16}.log")
run(["ncu","--set","basic","--launch-count","2","./native","profile","1024","4096","1","2"],"ncu-two-kernel-fp16.log")


In [ ]:
print(json.dumps({name:pathlib.Path(name).read_text() for name in ["ncu.log","native-build-final.log","ncu-fused-fp32.log","ncu-fused-fp16.log","ncu-two-kernel-fp16.log"]}))
